In [ ]:
!pip install scanpy
!pip install anndata
!pip3 install igraph
!pip install celltypist
!pip install decoupler
!pip install fa2-modified
!pip install louvain


In [ ]:
#Import core single cell datasets

import scanpy as sc
import anndata as ad
import numpy as np


In [ ]:
!wget https://ftp.ncbi.nlm.nih.gov/geo/series/GSE205nnn/GSE205790/suppl/GSE205790_RAW.tar

In [ ]:
!tar -xvf GSE205790_RAW.tar

In [ ]:

!mkdir -p GSM6229474
!mkdir -p GSM6229475
!mv GSM6229474_* GSM6229474
!mv GSM6229475_* GSM6229475

In [ ]:
!pwd

In [ ]:

!mv /content/GSM6229474/GSM6229474_C57IMQD_barcodes.tsv.gz /content/GSM6229474/barcodes.tsv.gz
!mv /content/GSM6229474/GSM6229474_C57IMQD_features.tsv.gz /content/GSM6229474/features.tsv.gz
!mv /content/GSM6229474/GSM6229474_C57IMQD_matrix.mtx.gz /content/GSM6229474/matrix.mtx.gz


!mv /content/GSM6229475/GSM6229475_C57IMQE_barcodes.tsv.gz /content/GSM6229475/barcodes.tsv.gz
!mv /content/GSM6229475/GSM6229475_C57IMQE_features.tsv.gz /content/GSM6229475/features.tsv.gz
!mv /content/GSM6229475/GSM6229475_C57IMQE_matrix.mtx.gz /content/GSM6229475/matrix.mtx.gz


In [ ]:
psorasis_mtx = sc.read_10x_mtx('GSM6229474/')
psorasis_mtx.var_names_make_unique()

In [ ]:
#psorasis_mtx = sc.read_10x_mtx('GSM6229475/')
#psorasis_mtx.var_names_make_unique()

In [ ]:
psorasis_mtx.shape

In [ ]:
psorasis_mtx.var.head()

In [ ]:
psorasis_mtx.var['MT'] = psorasis_mtx.var_names.str.startswith("MT-")
psorasis_mtx.var['RIBO'] = psorasis_mtx.var_names.str.startswith("RPS", "RPL")
psorasis_mtx.var['HB'] = psorasis_mtx.var_names.str.startswith("^HB[^(P)]")

In [ ]:
sc.pp.calculate_qc_metrics(
    psorasis_mtx, qc_vars=["MT", 'RIBO', 'HB'], inplace=True, log1p=True
)

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (5,4)  # Adjust figure size
plt.rcParams["axes.grid"] = True  # Add grid to plots
plt.rcParams["axes.edgecolor"] = "black" # Set plot border color
plt.rcParams["axes.linewidth"] = 1.5 # Set plot border width
plt.rcParams["axes.facecolor"] = "white" # Set background color
plt.rcParams["axes.labelcolor"] = "black" # Set label color
plt.rcParams["xtick.color"] = "black" # Set x-axis tick color
plt.rcParams["ytick.color"] = "black" # Set y-axis tick color
plt.rcParams["text.color"] = "black" # Set text color

In [ ]:
sc.pl.violin(
    psorasis_mtx,
    ["n_genes_by_counts", 'total_counts', 'pct_counts_MT'],
    jitter=0.4,
    multi_panel=False,
)

In [ ]:
sc.pl.scatter(psorasis_mtx, "total_counts", "n_genes_by_counts", color="pct_counts_MT")

In [ ]:
sc.pp.scrublet(psorasis_mtx)

In [ ]:
#Normalisation
psorasis_mtx.layers["counts"] = psorasis_mtx.X.copy()
sc.pp.normalize_total(psorasis_mtx)
sc.pp.log1p(psorasis_mtx)

In [ ]:
#Feature selection
sc.pp.highly_variable_genes(psorasis_mtx, n_top_genes=2500)
sc.pl.highly_variable_genes(psorasis_mtx)

In [ ]:
#Dim Reduction
sc.tl.pca(psorasis_mtx)
sc.pl.pca_variance_ratio(psorasis_mtx, n_pcs=10, log=False)


In [ ]:
il_genes = [gene for gene in psorasis_mtx.var_names[psorasis_mtx.var['highly_variable']] if gene.lower().startswith('il')]
print(il_genes)

In [ ]:
sc.pl.pca(psorasis_mtx, color="Il1b", cmap="coolwarm")

In [ ]:
sc.pp.neighbors(psorasis_mtx)
sc.tl.umap(psorasis_mtx)

In [ ]:
sc.pl.umap(
    psorasis_mtx,
    color=["Il1b"],
    size=8,
)

In [ ]:
sc.tl.leiden(psorasis_mtx, flavor="igraph", n_iterations=2, key_added="leiden_res0_5", resolution=0.05)

In [ ]:
sc.pl.umap(
    psorasis_mtx,
    color=["leiden_res0_5", 'predicted_doublet'],
    size=8,
)

In [ ]:
import decoupler as dc

In [ ]:
# Query Omnipath and get PanglaoDB
markers = dc.op.resource(name="PanglaoDB", organism="mouse")

# Keep canonical cell type markers alone
markers = markers[markers["mouse"]]

# Remove duplicated entries
markers = markers[~markers.duplicated(["cell_type", "genesymbol"])]

# Format because dc only accepts cell_type and genesymbol

markers = markers.rename(columns={"cell_type": "source", "genesymbol": "target"})
markers = markers[["source", "target"]]


markers.head()

In [ ]:
psorasis_mtx.var_names

In [ ]:
dc.mt.ulm(data=psorasis_mtx,
          net=markers,
          tmin = 3)

In [ ]:
score = dc.pp.get_obsm(psorasis_mtx, key="score_ulm")

In [ ]:
psorasis_mtx.obsm["score_ulm"].head(1)

In [ ]:
psorasis_mtx.obsm["score_ulm"].columns

In [ ]:
#rank genes
psorasis_gene_rank = dc.tl.rankby_group(score, groupby="leiden_res0_5", reference="rest", method="t-test_overestim_var")
psorasis_gene_rank = psorasis_gene_rank[psorasis_gene_rank["stat"] > 0]
psorasis_gene_rank.head(5)

In [ ]:
top_cell_type_per_group = psorasis_gene_rank.groupby('group')['name'].apply(lambda x: x.head(1))
display(top_cell_type_per_group.to_dict())

In [ ]:
sc.pl.umap(score, color=["Neutrophils","leiden_res0_5"], cmap="RdBu_r")

In [ ]:
dict_ann = psorasis_gene_rank[psorasis_gene_rank["stat"] > 0].groupby("group").head(1).set_index("group")["name"].to_dict()
dict_ann

In [ ]:
psorasis_mtx.obs["leiden_res0_5"] = psorasis_mtx.obs["leiden_res0_5"].cat.rename_categories(dict_ann)

In [ ]:
sc.pl.umap(
    adata=psorasis_mtx,
    color=[ "leiden_res0_5"],
    ncols=1,
)

In [ ]:
#Trajectory analysis
sc.tl.draw_graph(psorasis_mtx)

In [ ]:
plt.rcParams["figure.figsize"] = (4,4)
sc.pl.draw_graph(psorasis_mtx, color='leiden_res0_5')

In [ ]:
sc.tl.paga(psorasis_mtx, groups='leiden_res0_5')


In [ ]:
sc.pl.paga(psorasis_mtx, color=['leiden_res0_5'])

In [ ]:
sc.tl.draw_graph(psorasis_mtx, init_pos='paga')

In [ ]:
sc.pl.draw_graph(psorasis_mtx, color='leiden_res0_5', legend_loc='on data')

In [ ]:
sc.pl.paga_compare(psorasis_mtx, threshold=0.03, frameon=True, edges=True)

In [ ]:
psorasis_mtx.uns['iroot'] = np.flatnonzero(psorasis_mtx.obs['leiden_res0_5']  == 'T memory cells')[0]
sc.tl.dpt(psorasis_mtx)

In [ ]:
sc.pl.draw_graph(psorasis_mtx, color=['dpt_pseudotime'], legend_loc='on data')
